In [35]:
import pandas as pd
import torch

from torch.optim import Adam

from Ai.Behavior_Cloning.bc_lstm_training_pipeline import (
    build_dataloaders,
    build_model,
    train_model,
)

try:
    from Ai.Behavior_Cloning.action_masking_config import (
        WAIT_ID,
        ALWAYS_ALLOW_WAIT,
        AVAIL_FEATURE_TO_ACTION_ID,
        FULL_ELIXIR_THRESHOLD,
    )
except ImportError:
    # Supports running the notebook directly from Ai/Behavior_Cloning.
    from action_masking_config import (
        WAIT_ID,
        ALWAYS_ALLOW_WAIT,
        AVAIL_FEATURE_TO_ACTION_ID,
        FULL_ELIXIR_THRESHOLD,
    )


In [36]:
full_df = pd.read_csv('full_cleaned_dataset_lstm.csv')

features = [
    col for col in full_df.columns
    if col not in ['match_id', 'id', 'action', 'pos_x', 'pos_y']
]
targets = ['action', 'pos_x', 'pos_y']

NUM_ACTIONS = 13
WINDOW_SIZE = 10
BATCH_SIZE = 64


In [37]:
train_csv = r"C:\Users\SlayerDz\PycharmProjects\clash-royale-rl-agent\Ai\Behavior_Cloning\train_cleaned_dataset.csv"
val_csv = r"C:\Users\SlayerDz\PycharmProjects\clash-royale-rl-agent\Ai\Behavior_Cloning\test_cleaned_dataset.csv"

train_loaded, test_loaded = build_dataloaders(
    train_csv=train_csv,
    val_csv=val_csv,
    features=features,
    targets=targets,
    num_actions=NUM_ACTIONS,
    window_size=WINDOW_SIZE,
    batch_size=BATCH_SIZE,
    wait_id=WAIT_ID,
    avail_feature_to_action_id=AVAIL_FEATURE_TO_ACTION_ID,
    always_allow_wait=ALWAYS_ALLOW_WAIT,
)

print(f"train batches: {len(train_loaded)}")
print(f"val batches: {len(test_loaded)}")


train batches: 33
val batches: 15


In [38]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = build_model(
    input_size=len(features),
    hidden_size=128,
    num_layers=2,
    num_actions=NUM_ACTIONS,
    dropout=0.1,
    device=device,
)

optimizer = Adam(model.parameters(), lr=1e-3)


In [39]:
history = train_model(
    model=model,
    train_loader=train_loaded,
    val_loader=test_loaded,
    device=device,
    optimizer=optimizer,
    num_epochs=50,
    pos_weight=0.7,
    wait_id=WAIT_ID,
    wait_weight=0.1,
    leak_weight=0.2,
    full_elixir_threshold=FULL_ELIXIR_THRESHOLD,
    log_file='training_metrics.jsonl',
)


Epoch 001  train_loss=13.9619 train_acc=0.2506 train_leak=0.0122 val_loss=5.1796 val_acc=0.2331 val_leak=0.0000
Epoch 002  train_loss=5.2528 train_acc=0.2774 train_leak=0.0122 val_loss=4.8428 val_acc=0.2691 val_leak=0.0208
Epoch 003  train_loss=5.1119 train_acc=0.3036 train_leak=0.0488 val_loss=5.2196 val_acc=0.2225 val_leak=0.0000
Epoch 004  train_loss=5.0204 train_acc=0.2886 train_leak=0.0000 val_loss=5.1781 val_acc=0.2331 val_leak=0.0000
Epoch 005  train_loss=4.8584 train_acc=0.2754 train_leak=0.0000 val_loss=5.6363 val_acc=0.2701 val_leak=0.0208
Epoch 006  train_loss=4.7382 train_acc=0.3002 train_leak=0.0000 val_loss=4.9818 val_acc=0.2044 val_leak=0.0000
Epoch 007  train_loss=4.4498 train_acc=0.2837 train_leak=0.0000 val_loss=5.2892 val_acc=0.2383 val_leak=0.0000
Epoch 008  train_loss=4.2950 train_acc=0.2754 train_leak=0.0000 val_loss=5.5312 val_acc=0.2362 val_leak=0.0000
Epoch 009  train_loss=3.9582 train_acc=0.3027 train_leak=0.0000 val_loss=5.3492 val_acc=0.2129 val_leak=0.0000


In [40]:
torch.save(model.state_dict(), 'lstm.pth')
print('Model saved to lstm.pth')


Model saved to lstm.pth


In [41]:
# Optional: inspect final validation metrics
if history['val']:
    print(history['val'][-1])


{'event': 'epoch', 'epoch': 50, 'split': 'val', 'total_loss': 5.9429231700250655, 'action_loss': 2.0120712700536694, 'pos_loss': 5.506153100241314, 'leak_loss': 0.1322027960066068, 'action_acc': 0.3411016949152542, 'pos_mae': 1.8106394264914774, 'leak_rate_at_full_elixir': 0.0625, 'total_samples': 944, 'valid_pos_samples': 275, 'valid_pos_coords': 550, 'correct_actions': 322, 'leak_eval_samples': 48, 'leak_pred_wait': 3, 'epoch_seconds': 0.24175143241882324, 'timestamp': 1776453911.7427852}


In [42]:
# Optional: save history as a CSV for quick plotting
history_df = pd.DataFrame(history['train'])
history_df['split'] = 'train'
val_df = pd.DataFrame(history['val'])
val_df['split'] = 'val'
all_hist = pd.concat([history_df, val_df], ignore_index=True)
all_hist.to_csv('training_history.csv', index=False)
print('Saved training_history.csv')


Saved training_history.csv


In [43]:
# Example: best epoch by validation action accuracy
if history['val']:
    best = max(history['val'], key=lambda x: x['action_acc'])
    print('Best val action_acc epoch:', best['epoch'])
    print('Best val action_acc:', best['action_acc'])


Best val action_acc epoch: 49
Best val action_acc: 0.4067796610169492


In [44]:
# Example: best epoch by lowest full-elixir leak rate
if history['val']:
    best_leak = min(history['val'], key=lambda x: x['leak_rate_at_full_elixir'])
    print('Best leak epoch:', best_leak['epoch'])
    print('Best leak_rate_at_full_elixir:', best_leak['leak_rate_at_full_elixir'])


Best leak epoch: 1
Best leak_rate_at_full_elixir: 0.0


In [45]:
# Notes:
# - Action masking is handled inside bc_lstm_training_pipeline.py
# - wait is always allowed as fallback
# - leak penalty discourages waiting at full elixir when legal non-wait actions exist
